In [9]:
import json

with open('medicines.json', 'r') as file:
    data = json.load(file)




In [10]:
len(data)

18

In [11]:
data[0]

{'drug_name': 'Metformin',
 'generic_name': 'metformin hydrochloride',
 'drug_class': 'Biguanide (oral antidiabetic; non-sulfonylurea)',
 'uses': ['Treatment of type 2 diabetes mellitus to control high blood sugar, together with diet and exercise',
  'Used alone or in combination with other diabetes medicines, including insulin'],
 'indications': ['Type 2 diabetes mellitus (adjunct to diet and exercise to improve glycemic control)',
  'Off-label per Drugs.com treatment guides: polycystic ovary syndrome (PCOS) and insulin resistance'],
 'common_side_effects': ['Diarrhea',
  'Nausea',
  'Upset stomach / stomach discomfort',
  'Vomiting',
  'Gas (flatulence)',
  'Indigestion',
  'Constipation',
  'Metallic or altered taste',
  'Headache',
  'Weakness or lack of energy',
  'Muscle pain',
  'Flushing of the skin',
  'Low blood sugar (mainly when combined with insulin or sulfonylureas)'],
 'serious_side_effects': ['Lactic acidosis (rare, life-threatening): unusual muscle pain, trouble breath

In [ ]:
# from langchain_core.documents import Document

# def make_chunks(drug):
#     name = drug["drug_name"]
#     prefix = f"Drug: {name} ({drug['drug_class']})"
  
#     meta = {"drug_name": name}
    
#     chunks = [
#         Document(page_content=f"{prefix} | Uses: {drug['uses']} | Indications: {drug['indications']}", metadata={**meta, "chunk_type": "uses"}),
#         Document(page_content=f"{prefix} | Common side effects: {drug['common_side_effects']} | Serious: {drug['serious_side_effects']}", metadata={**meta, "chunk_type": "side_effects"}),
#         Document(page_content=f"{prefix} | Warnings: {drug['warnings']} | Contraindications: {drug['contraindications']} | Disease interactions: {drug['drug_disease_interactions']}", metadata={**meta, "chunk_type": "warnings"}),
#         Document(page_content=f"{prefix} | Dose: {drug['usual_adult_dose']} | Frequency: {drug['usual_adult_frequency']} | Administration: {drug['administration_instructions']} | Missed dose: {drug['missed_dose_instructions']} | Storage: {drug['storage_instructions']}", metadata={**meta, "chunk_type": "dosage"}),
#         Document(page_content=f"{prefix} | Drug interactions: {drug['major_drug_interactions']} | Food interactions: {drug['drug_food_interactions']}", metadata={**meta, "chunk_type": "interactions"}),
#         Document(page_content=f"{prefix} | Monitoring: {drug['monitoring_requirements']} | Counseling: {drug['patient_counseling_points']}", metadata={**meta, "chunk_type": "monitoring"}),
#     ]
    
#     for symptom in drug.get("symptom_related_information", []):
#         chunks.append(Document(
#             page_content=f"{prefix} | Symptom: {symptom['symptom']} | {symptom['possible_relation_to_drug']}",
#             metadata={**meta, "chunk_type": "symptom"}
#         ))
    
#     return chunks

In [32]:
from langchain_core.documents import Document

def make_chunks(drug):
    name = drug["drug_name"]
    prefix = f"Drug: {name} ({drug['drug_class']})"
    sources = json.dumps(drug.get("sources", []))
    meta = {"drug_name": name, "sources": sources}

    sources = drug.get("sources", [])

    uses = "\n- ".join(drug['uses'])
    indications = "\n- ".join(drug['indications'])
    common_side_effects = "\n- ".join(drug['common_side_effects'])
    serious_side_effects = "\n- ".join(drug['serious_side_effects'])
    warnings = "\n- ".join(drug['warnings'])
    contraindications = "\n- ".join(drug['contraindications'])
    monitoring_requirements = "\n- ".join(drug['monitoring_requirements'])
    patient_counseling_points = "\n- ".join(drug['patient_counseling_points'])

    ## dict inside list

    disease_interactions = "\n- ".join(f"{d['disease_or_condition']}: {d['description']}" for d in drug['drug_disease_interactions'])
    drug_interactions = "\n- ".join(f"{d['interacting_drug']}: {d['description']}" for d in drug['major_drug_interactions'])
    food_interactions = "\n- ".join(f"{d['food_or_beverage']}: {d['description']}" for d in drug['drug_food_interactions'])

    chunks = [
        Document(page_content=f"{prefix} | Uses: {uses} | Indications: {indications}", metadata={**meta, "chunk_type": "uses"}),
        Document(page_content=f"{prefix} | Common side effects: {common_side_effects} | Serious: {serious_side_effects}", metadata={**meta, "chunk_type": "side_effects"}),
        Document(page_content=f"{prefix} | Warnings: {warnings} | Contraindications: {contraindications} | Disease interactions: {disease_interactions}", metadata={**meta, "chunk_type": "warnings"}),
        Document(page_content=f"{prefix} | Dose: {drug['usual_adult_dose']} | Frequency: {drug['usual_adult_frequency']} | Administration: {drug['administration_instructions']} | Missed dose: {drug['missed_dose_instructions']} | Storage: {drug['storage_instructions']}", metadata={**meta, "chunk_type": "dosage"}),
        Document(page_content=f"{prefix} | Drug interactions: {drug_interactions} | Food interactions: {food_interactions}", metadata={**meta, "chunk_type": "interactions"}),
        Document(page_content=f"{prefix} | Monitoring: {monitoring_requirements} | Counseling: {patient_counseling_points}", metadata={**meta, "chunk_type": "monitoring"}),
    ]

    for symptom in drug.get("symptom_related_information", []):
        chunks.append(Document(
            page_content=f"{prefix} | Symptom: {symptom['symptom']} | {symptom['possible_relation_to_drug']}",
            metadata={**meta, "chunk_type": "symptom"}
        ))

    return chunks

In [33]:
chunks = []
for drug in data:
    chunks.extend(make_chunks(drug))


In [34]:
len(chunks)

164

In [35]:
chunks

[Document(metadata={'drug_name': 'Metformin', 'sources': '[{"source_name": "MedlinePlus (NLM)", "url": "https://medlineplus.gov/druginfo/meds/a696005.html"}, {"source_name": "Drugs.com", "url": "https://www.drugs.com/metformin.html"}, {"source_name": "Drugs.com - Dosage", "url": "https://www.drugs.com/dosage/metformin.html"}]', 'chunk_type': 'uses'}, page_content='Drug: Metformin (Biguanide (oral antidiabetic; non-sulfonylurea)) | Uses: Treatment of type 2 diabetes mellitus to control high blood sugar, together with diet and exercise\n- Used alone or in combination with other diabetes medicines, including insulin | Indications: Type 2 diabetes mellitus (adjunct to diet and exercise to improve glycemic control)\n- Off-label per Drugs.com treatment guides: polycystic ovary syndrome (PCOS) and insulin resistance'),
 Document(metadata={'drug_name': 'Metformin', 'sources': '[{"source_name": "MedlinePlus (NLM)", "url": "https://medlineplus.gov/druginfo/meds/a696005.html"}, {"source_name": "D

In [37]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

In [38]:
vdb = Chroma.from_documents(
  documents=chunks,
  embedding=embeddings,
  persist_directory="./chroma_db"
  )